# Nonlinear Hawkes: 4D Parameter Space Search for Mean-Field Mismatch

Searches over **(α, b, μ, τ)** using Latin Hypercube Sampling (50 000 analytical samples),
then simulates the top 15 candidates to measure true stochastic mismatch.

**Model (current-based synapse):**
$$v_{j+1} = v_j + \frac{\mathrm{d}t\,(-v_j+\mu) + \alpha\,\mathrm{spike}_j}{\tau}, \quad
\mathrm{spike}_j \sim \mathrm{Bernoulli}(\phi(v_j)\,\mathrm{d}t), \quad \phi(v)=\max(v,0)^2/b$$

Mean-field fixed point: $\bar v^* = \mu + \alpha\,\phi(\bar v^*)$  
Stability: $\lambda = \bigl(-1 + \alpha\,\phi'(\bar v^*)\bigr)/\tau < 0$  
Noise amplitude: $D_0 = \alpha^2\,\phi(\bar v^*)$ (current-based: each spike injects fixed $\alpha$)  
Predicted variance: $\mathrm{Var}(v)\approx D_0/(2|\lambda|\tau)$  
Predicted relative rate mismatch (quadratic only): $\Delta = \mathrm{Var}(v)/\bar v^{*2}$

**Requirements:**
- $v \geq 0$ always (hard clamp), threshold rarely hit: safety $= \bar v^*/\sigma_v > 3$
- Consistent spiking: $\phi(\bar v^*) > 1$ and $\phi(\bar v^* - 2\sigma_v) > 0.3$
- Quadratic regime: spike jump $\alpha/\tau \ll \bar v^*$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.stats.qmc import LatinHypercube
import warnings
warnings.filterwarnings('ignore')

# ── model functions (current-based synapse, no E) ─────────────────────────────
def phi(v, b):       return np.maximum(v, 0.0)**2 / b
def phi_prime(v, b): return 2.0 * np.maximum(v, 0.0) / b

def find_stable_fp(alpha, b, mu, tau):
    """Stable positive fixed point of  dv/dt = (-v + mu)/tau + alpha*phi(v)/tau."""
    def rhs(v): return (-v + mu) / tau + alpha * phi(v, b) / tau
    vhi = max(mu * 30, 100.0)
    vs  = np.linspace(1e-4, vhi, 8000)
    fs  = rhs(vs)
    for i in range(len(vs) - 1):
        if fs[i] * fs[i+1] < 0:
            try:
                vbar = brentq(rhs, vs[i], vs[i+1], xtol=1e-12)
                lam  = (-1.0 + alpha * phi_prime(vbar, b)) / tau
                if lam < 0 and vbar > 0:
                    return vbar, lam
            except Exception:
                pass
    return None

print('Model functions defined (current-based, no conductance term).')

## 1. Latin Hypercube Sampling — analytical scan (50 000 samples)

In [ ]:
N_SAMPLES = 50_000

# 4 parameters only: [alpha, b, mu, tau]  — no E
lo = np.array([0.02, 0.1,  0.5, 0.1])
hi = np.array([2.00, 8.0,  6.0, 5.0])

sampler = LatinHypercube(d=4, seed=0)
unit    = sampler.random(n=N_SAMPLES)
params  = lo + (hi - lo) * unit

# ── filter constants ──────────────────────────────────────────────────────────
SAFETY_MIN   = 3.0   # v̄*/√Var(v) > 3  →  P(v<0) < 0.13%
SAFETY_MAX   = 8.0   # ensures measurable mismatch
DELTA_MIN    = 0.01
RATE_MIN     = 1.0   # φ(v̄*) > 1 spk/s
RATE_MAX     = 500.0
RATE_LOW_MIN = 0.3   # φ(v̄* − 2σ) > 0.3  (consistent spiking)
JUMP_MAX     = 0.3   # α/τ  <  JUMP_MAX * v̄*  (Langevin approx valid)

results = []
counters = dict(no_fp=0, bad_rate=0, jump_too_large=0,
                threshold_dominated=0, mismatch_too_small=0,
                inconsistent_spiking=0)

for alpha, b, mu, tau in params:
    fp = find_stable_fp(alpha, b, mu, tau)
    if fp is None:
        counters['no_fp'] += 1; continue
    vbar, lam = fp
    r = phi(vbar, b)

    if r < RATE_MIN or r > RATE_MAX:
        counters['bad_rate'] += 1; continue

    # jump-size: fixed current injection α/τ per spike
    jump = alpha / tau
    if jump > JUMP_MAX * vbar:
        counters['jump_too_large'] += 1; continue

    # noise: D0 = α² * φ(v̄*)  (current-based)
    D0    = alpha**2 * r
    var_v = D0 / (2.0 * abs(lam) * tau)
    std_v = np.sqrt(var_v)
    safety = vbar / std_v if std_v > 0 else np.inf

    if safety < SAFETY_MIN:
        counters['threshold_dominated'] += 1; continue
    if safety > SAFETY_MAX:
        counters['mismatch_too_small'] += 1; continue

    delta = var_v / vbar**2
    if delta < DELTA_MIN:
        counters['mismatch_too_small'] += 1; continue

    v_low    = max(0.0, vbar - 2.0 * std_v)
    rate_low = phi(v_low, b)
    if rate_low < RATE_LOW_MIN:
        counters['inconsistent_spiking'] += 1; continue

    results.append(dict(alpha=alpha, b=b, mu=mu, tau=tau,
                        vbar=vbar, lam=lam, rate_mf=r, rate_low=rate_low,
                        var_v=var_v, std_v=std_v, safety=safety,
                        delta=delta, jump=jump))

print(f'Total samples:                        {N_SAMPLES:>8}')
for k, v in counters.items():
    print(f'  Rejected ({k:<28}): {v:>8}')
print(f'PASSED all filters:                   {len(results):>8}')

results.sort(key=lambda r: r['delta'], reverse=True)
top15 = results[:15]

print(f'\nTop 20 by Δ_quad = Var(v)/v̄*²:')
print(f'{"α":>7} {"b":>6} {"μ":>6} {"τ":>6}  {"v̄*":>7} {"λ":>9} '
      f'{"φ(v̄*)":>8} {"φ(v̄*−2σ)":>10} {"safety":>7} {"jump/v̄*":>9} {"Δ_quad":>8}')
print('-'*90)
for r in results[:20]:
    print(f"{r['alpha']:7.3f} {r['b']:6.3f} {r['mu']:6.3f} {r['tau']:6.3f}  "
          f"{r['vbar']:7.3f} {r['lam']:9.4f} {r['rate_mf']:8.3f} "
          f"{r['rate_low']:10.3f} {r['safety']:7.2f} "
          f"{r['jump']/r['vbar']:9.3f} {r['delta']:8.5f}")

## 2. Visualise the analytical landscape

In [ ]:
arr = np.array([[r['alpha'], r['b'], r['mu'], r['tau'],
                  r['delta'], r['safety']] for r in results])
alpha_a, b_a, mu_a, tau_a, delta_a, safety_a = arr.T
log_d = np.log10(np.clip(delta_a, 1e-6, None))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

pairs = [(alpha_a, b_a,   'α', 'b'),
         (mu_a,   tau_a,  'μ', 'τ'),
         (alpha_a, mu_a,  'α', 'μ'),
         (b_a,    tau_a,  'b', 'τ')]

key_map = {'α': alpha_a, 'b': b_a, 'μ': mu_a, 'τ': tau_a}
r_key   = {'α': 'alpha', 'b': 'b', 'μ': 'mu', 'τ': 'tau'}

# top row: colored by Δ_quad
for ax, (x, y, xl, yl) in zip(axes[0], pairs):
    sc = ax.scatter(x, y, c=log_d, s=3, alpha=0.4, cmap='plasma')
    plt.colorbar(sc, ax=ax, label='log₁₀(Δ_quad)')
    for r in top15:
        ax.plot(r[r_key[xl]], r[r_key[yl]], 'r*', ms=10)
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.set_title('colored by Δ_quad')

# bottom row: colored by safety ratio
for ax, (x, y, xl, yl) in zip(axes[1], pairs):
    sc = ax.scatter(x, y, c=safety_a, s=3, alpha=0.4, cmap='viridis',
                    vmin=SAFETY_MIN, vmax=SAFETY_MAX)
    plt.colorbar(sc, ax=ax, label='safety = v̄*/σ_v')
    for r in top15:
        ax.plot(r[r_key[xl]], r[r_key[yl]], 'r*', ms=10)
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.set_title('colored by safety ratio')

fig.suptitle(f'Passing samples (safety ∈ [{SAFETY_MIN},{SAFETY_MAX}], quadratic regime only)\n'
             f'Red stars = top-15 candidates', fontsize=12)
plt.tight_layout()
plt.show()

fig, axes2 = plt.subplots(1, 2, figsize=(11, 4))
axes2[0].hist(log_d, bins=60, color='steelblue', edgecolor='none')
axes2[0].set_xlabel('log₁₀(Δ_quad)'); axes2[0].set_ylabel('count')
axes2[0].set_title('Distribution of quadratic mismatch')
axes2[1].hist(safety_a, bins=60, color='C2', edgecolor='none')
axes2[1].axvline(SAFETY_MIN, color='r', ls='--', label=f'min={SAFETY_MIN}')
axes2[1].axvline(SAFETY_MAX, color='r', ls='--', label=f'max={SAFETY_MAX}')
axes2[1].set_xlabel('safety = v̄*/√Var(v)'); axes2[1].set_ylabel('count')
axes2[1].set_title('Distribution of safety ratio')
axes2[1].legend()
plt.tight_layout()
plt.show()

## 3. Stochastic simulation of top-15 candidates

In [ ]:
T_SIM    = 2000.0
DT_SIM   = 5e-3
BURNIN   = 500.0
N_TRIALS = 6
BURNIN_IDX = int(BURNIN / DT_SIM)

def simulate_trial(alpha, b, mu, tau, T=T_SIM, dt=DT_SIM, x0=1.0, seed=0):
    """Current-based Hawkes neuron. v cannot go below 0."""
    rng = np.random.default_rng(seed)
    N   = int(np.floor(T / dt))
    v   = np.empty(N)
    v[0] = max(0.0, x0)
    spk = np.zeros(N, dtype=np.int8)
    for j in range(N - 1):
        lam = min(phi(v[j], b), 1e8)
        spk[j] = 1 if rng.random() < lam * dt else 0
        v_new = v[j] + (dt * (-v[j] + mu) + alpha * spk[j]) / tau
        v[j+1] = max(0.0, v_new)   # hard clamp
        if not np.isfinite(v[j+1]) or v[j+1] > 1e6:
            return None, None
    return v, spk

def run_mean_field(alpha, b, mu, tau, T=T_SIM, dt=DT_SIM, x0=None):
    N = int(np.floor(T / dt))
    vbar = np.empty(N)
    vbar[0] = max(0.0, x0 if x0 is not None else mu)
    for j in range(N - 1):
        vbar[j+1] = max(0.0, vbar[j] + (dt * (-vbar[j] + mu) + alpha * phi(vbar[j], b) * dt) / tau)
    return np.arange(N) * dt, vbar

sim_results = []

for ci, r in enumerate(top15):
    alpha, b, mu, tau = r['alpha'], r['b'], r['mu'], r['tau']
    x0 = r['vbar']
    v_means, rate_means, frac_zero = [], [], []
    stable = True

    for seed in range(N_TRIALS):
        v, spk = simulate_trial(alpha, b, mu, tau, x0=x0, seed=seed)
        if v is None:
            stable = False; break
        vp = v[BURNIN_IDX:]
        v_means.append(np.mean(vp))
        rate_means.append(np.mean(phi(vp, b)))
        frac_zero.append(np.mean(vp == 0.0))

    if not stable:
        sim_results.append({**r, 'stable': False})
        print(f'  candidate {ci+1:2d}/15 — UNSTABLE')
        continue

    vm = np.mean(v_means)
    rm = np.mean(rate_means)
    fz = np.mean(frac_zero)
    pv = 100 * (vm - r['vbar'])    / abs(r['vbar'])
    pr = 100 * (rm - r['rate_mf']) / abs(r['rate_mf'])
    sim_results.append({**r, 'stable': True,
                        'v_emp': vm, 'rate_emp': rm,
                        'frac_zero': fz, 'pct_v': pv, 'pct_rate': pr})
    if (ci + 1) % 5 == 0 or ci == 0:
        print(f'  candidate {ci+1:2d}/15  rate mismatch {pr:+.1f}%  frac_at_clamp={fz:.5f}')

print('\nSimulation complete.')

## 4. Results table

In [ ]:
stable_sims = [r for r in sim_results if r.get('stable')]
stable_sims.sort(key=lambda r: abs(r['pct_rate']), reverse=True)

print('='*105)
print('RESULTS — ranked by |measured rate mismatch|')
print('='*105)
hdr = (f'{"":3}{"α":>7}{"b":>6}{"μ":>6}{"τ":>6}  {"v̄*":>7}{"λ":>9}'
       f'{"φ(v̄*)":>8}{"Δ_quad":>8}  {"⟨v⟩":>8}{"v%":>7}  {"⟨φ(v)⟩":>9}{"rate%":>7}  {"@clamp":>7}')
print(hdr)
print('-'*105)
for i, r in enumerate(stable_sims):
    star = '★  ' if i < 3 else '   '
    print(f"{star}{r['alpha']:7.3f}{r['b']:6.3f}{r['mu']:6.3f}{r['tau']:6.3f}  "
          f"{r['vbar']:7.3f}{r['lam']:9.4f}{r['rate_mf']:8.3f}{r['delta']:8.5f}  "
          f"{r['v_emp']:8.4f}{r['pct_v']:+7.1f}%  "
          f"{r['rate_emp']:9.4f}{r['pct_rate']:+7.1f}%  "
          f"{r['frac_zero']:6.4f}")
print('='*105)
print('(@clamp = fraction of timesteps where v hit the v=0 floor — should be < 0.001)')

best = stable_sims[0]
print(f"\n★ BEST: α={best['alpha']:.3f}, b={best['b']:.3f}, μ={best['mu']:.3f}, τ={best['tau']:.3f}")
print(f"  v̄* = {best['vbar']:.4f},  λ = {best['lam']:.4f},  φ(v̄*) = {best['rate_mf']:.4f}")
print(f"  Measured: ⟨v⟩ = {best['v_emp']:.4f}  ({best['pct_v']:+.2f}%),  "
      f"⟨φ(v)⟩ = {best['rate_emp']:.4f}  ({best['pct_rate']:+.2f}%)")
print(f"  Fraction at v=0 clamp: {best['frac_zero']:.5f}")

## 5. Detailed plots — best parameter set

In [ ]:
alpha, b, mu, tau = best['alpha'], best['b'], best['mu'], best['tau']
vbar_star = best['vbar']

v_long, spk_long = simulate_trial(alpha, b, mu, tau, T=T_SIM, dt=DT_SIM, x0=vbar_star, seed=99)
N_sim = len(v_long)
t_sim, vbar_ode = run_mean_field(alpha, b, mu, tau, T=T_SIM, dt=DT_SIM, x0=vbar_star)

sigma_s = 0.2; sb = sigma_s / DT_SIM
hw = int(np.ceil(4 * sb)); xk = np.arange(-hw, hw + 1)
g  = np.exp(-0.5 * (xk / sb)**2); g /= g.sum() * DT_SIM
rate_emp_sm = np.convolve(spk_long.astype(float), g, mode='same')
rate_mf_sm  = phi(vbar_ode, b)

SHOW = int(300 / DT_SIM)
t0  = t_sim[BURNIN_IDX:BURNIN_IDX+SHOW] - t_sim[BURNIN_IDX]
v0  = v_long[BURNIN_IDX:BURNIN_IDX+SHOW]
vb0 = vbar_ode[BURNIN_IDX:BURNIN_IDX+SHOW]
r0  = rate_emp_sm[BURNIN_IDX:BURNIN_IDX+SHOW]
rm0 = rate_mf_sm[BURNIN_IDX:BURNIN_IDX+SHOW]

D0    = alpha**2 * phi(vbar_star, b)
var_v = D0 / (2 * abs(best['lam']) * tau)

param_str = f'α={alpha:.3f}, b={b:.3f}, μ={mu:.3f}, τ={tau:.3f}'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.plot(t0, v0, lw=0.5, alpha=0.7, label='stochastic v(t)')
ax.plot(t0, vb0, lw=2, color='C1', label='mean-field v̄(t)')
ax.axhline(vbar_star, ls='--', color='k', lw=1.2, label=f'v̄*={vbar_star:.3f}')
ax.axhline(0, ls=':', color='gray', lw=0.8, label='v=0 floor')
ax.set_ylabel('v'); ax.set_xlabel('t (s)'); ax.legend(fontsize=8)
ax.set_title(f'Voltage  |  {param_str}')

ax = axes[0, 1]
ax.plot(t0, r0,  lw=0.8, alpha=0.8, label=f'empirical rate (σ={sigma_s*1e3:.0f} ms)')
ax.plot(t0, rm0, lw=2, color='C1', label='mean-field φ(v̄)')
ax.axhline(best['rate_mf'], ls='--', color='k', lw=1.2, label=f'φ(v̄*)={best["rate_mf"]:.3f}')
ax.set_ylabel('rate (spk/s)'); ax.set_xlabel('t (s)'); ax.legend(fontsize=8)
ax.set_title('Firing rate')

ax = axes[1, 0]
v_post = v_long[BURNIN_IDX:]
ax.hist(v_post, bins=80, density=True, color='steelblue', alpha=0.7, label='empirical')
vgrid = np.linspace(0, v_post.max(), 400)
gauss = np.exp(-0.5 * (vgrid - vbar_star)**2 / var_v) / np.sqrt(2 * np.pi * var_v)
ax.plot(vgrid, gauss, 'C1', lw=2, label=f'Gaussian(v̄*, σ={np.sqrt(var_v):.3f})')
ax.axvline(vbar_star, color='k', ls='--', lw=1.2, label=f'v̄*={vbar_star:.3f}')
ax.axvline(np.mean(v_post), color='C0', ls=':', lw=1.5, label=f'⟨v⟩={np.mean(v_post):.3f}')
ax.set_xlabel('v'); ax.set_ylabel('density'); ax.legend(fontsize=8)
ax.set_title('Stationary distribution of v')

ax = axes[1, 1]
vr = np.linspace(0, vbar_star * 2.5 + 0.5, 500)
dvdt_mf = (-vr + mu) / tau + alpha * phi(vr, b) / tau
ax.plot(vr, dvdt_mf, 'k', lw=2.5, label='mean-field dv/dt', zorder=5)
ax.axhline(0, color='gray', lw=0.8)
ax.axvline(vbar_star, color='C2', ls='--', lw=1.5, label=f'v̄*={vbar_star:.3f}')
dv_stoch = np.diff(v0) / DT_SIM
ax.scatter(v0[:-1], dv_stoch, s=0.4, alpha=0.15, color='C0', label='stochastic (v, dv/dt)')
ax.set_xlabel('v'); ax.set_ylabel('dv/dt'); ax.legend(fontsize=8)
ax.set_title('Phase portrait')
ylim = np.percentile(np.abs(dv_stoch), 99) * 1.5
ax.set_ylim(-ylim, ylim)

plt.suptitle(f'Best parameter set  |  rate mismatch {best["pct_rate"]:+.1f}%', fontsize=13)
plt.tight_layout()
plt.show()

## 6. 2D heatmaps around the best point

In [ ]:
NGRID = 30
p0 = dict(alpha=best['alpha'], b=best['b'], mu=best['mu'], tau=best['tau'])

def rng2(key, lo_b, hi_b):
    c = p0[key]
    return np.linspace(max(lo_b, c * 0.4), min(hi_b, c * 1.6), NGRID)

pair_specs = [
    ('alpha', 'b',   0.02, 2.0, 0.1, 8.0),
    ('mu',   'tau',  0.5,  6.0, 0.1, 5.0),
    ('alpha', 'mu',  0.02, 2.0, 0.5, 6.0),
    ('b',    'tau',  0.1,  8.0, 0.1, 5.0),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 11))

for ax, (k1, k2, lo1, hi1, lo2, hi2) in zip(axes.flat, pair_specs):
    x_arr = rng2(k1, lo1, hi1)
    y_arr = rng2(k2, lo2, hi2)
    Z = np.full((NGRID, NGRID), np.nan)
    for xi, xv in enumerate(x_arr):
        for yi, yv in enumerate(y_arr):
            kw = dict(**p0, **{k1: xv, k2: yv})
            fp = find_stable_fp(kw['alpha'], kw['b'], kw['mu'], kw['tau'])
            if fp is None: continue
            vb, lm = fp
            D0 = kw['alpha']**2 * phi(vb, kw['b'])
            var_v = D0 / (2 * abs(lm) * kw['tau'])
            d = var_v / vb**2
            if np.isfinite(d) and d < 20:
                Z[yi, xi] = np.log10(max(d, 1e-6))
    im = ax.imshow(Z, origin='lower', aspect='auto',
                   extent=[x_arr[0], x_arr[-1], y_arr[0], y_arr[-1]],
                   cmap='viridis')
    plt.colorbar(im, ax=ax, label='log₁₀(Δ_quad)')
    ax.plot(p0[k1], p0[k2], 'r*', ms=15, label='best')
    ax.set_xlabel(k1); ax.set_ylabel(k2); ax.legend(fontsize=8)

fig.suptitle('Predicted Δ_quad — 2D slices around best point\n(other 2 params fixed at best values)', fontsize=12)
plt.tight_layout()
plt.show()